# ClinVar Data Analysis

Step-by-step notebook: **import → clean → analyze → visualize** the ClinVar `variant_summary` file (~9 million rows).

Run the cells top to bottom (`Runtime → Run all`). Each step says what it does.

> Scope reminder: we study patterns in the dataset. We do **not** diagnose anyone or change expert labels.

## Step 0 · Setup

In [ ]:
import os, gc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)

# Check how much memory this Colab machine has (free tier is ~12 GB)
!free -h | head -2

Optional: save files to Google Drive so you don't re-download every session.
Set `USE_DRIVE = True` to turn it on.

In [ ]:
USE_DRIVE = False

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = "/content/drive/MyDrive/clinvar"
else:
    DATA_DIR = "/content/clinvar"
os.makedirs(DATA_DIR, exist_ok=True)

RAW_FILE = f"{DATA_DIR}/variant_summary.txt.gz"
CLEAN_FILE = f"{DATA_DIR}/clinvar_grch38_clean.parquet"

## Step 1 · Import (download the data)

Downloads the official file from NCBI (~400 MB compressed). We keep it compressed; pandas can read `.gz` directly.

In [ ]:
URL = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz"
if not os.path.exists(RAW_FILE):
    !wget -q --show-progress -O "{RAW_FILE}" "{URL}"
!ls -lh "{RAW_FILE}"

### 1b · Peek at the first rows (cheap: reads only 5 rows)

In [ ]:
peek = pd.read_csv(RAW_FILE, sep="\t", nrows=5, dtype=str)
print(len(peek.columns), "columns:")
print(list(peek.columns))
peek.head()

## Step 2 · Load only what we need

The full file does not fit comfortably in memory. So we:
1. read only the columns we need,
2. read the file in chunks of 500,000 rows,
3. keep only **GRCh38** rows (this also removes the duplicate copy of each variant).

In [ ]:
COLS = ["VariationID", "Type", "GeneSymbol", "ClinicalSignificance", "ReviewStatus",
        "NumberSubmitters", "PhenotypeList", "Assembly", "Chromosome", "Start",
        "LastEvaluated", "OriginSimple"]

missing = [c for c in COLS if c not in peek.columns]
assert not missing, f"Column names changed in this ClinVar release: {missing}"

parts, total = [], 0
reader = pd.read_csv(RAW_FILE, sep="\t", usecols=COLS, dtype=str,
                     chunksize=500_000, quoting=3)  # quoting=3: treat quote marks as plain text
for chunk in reader:
    total += len(chunk)
    parts.append(chunk[chunk["Assembly"] == "GRCh38"])
raw = pd.concat(parts, ignore_index=True)
del parts; gc.collect()

print(f"Rows in file:      {total:,}")
print(f"Rows kept (GRCh38): {len(raw):,}")

## Step 3 · Clean

### 3a · Make "missing" mean one thing
ClinVar writes missing data as `-`, `na`, `not provided`, `not specified` or `-1`. We turn all of them into a real empty value (`NaN`).

In [ ]:
df = raw
del raw  # one copy in memory, not two
MISSING = ["-", "na", "NA", "", "not provided", "not specified", "-1"]

before = df.isna().sum()
df = df.replace(MISSING, np.nan)
print(pd.DataFrame({"missing_before": before, "missing_after": df.isna().sum()}))

### 3b · Fix data types (numbers and dates)

In [ ]:
df["Start"] = pd.to_numeric(df["Start"], errors="coerce").astype("Int64")
df["NumberSubmitters"] = pd.to_numeric(df["NumberSubmitters"], errors="coerce").astype("Int64")

# Dates look like "Mar 05, 2019". Anything else becomes empty (NaT).
df["LastEvaluated"] = pd.to_datetime(df["LastEvaluated"], format="%b %d, %Y", errors="coerce")
df["EvalYear"] = df["LastEvaluated"].dt.year.astype("Int64")
df[["Start", "NumberSubmitters", "LastEvaluated", "EvalYear"]].describe(include="all")

### 3c · Remove exact duplicates

In [ ]:
n = len(df)
df = df.drop_duplicates(subset=["VariationID"])
print(f"Removed {n - len(df):,} duplicate rows. Now {len(df):,} unique variants.")

### 3d · Group the messy significance labels
There are hundreds of label combinations. We map them into 5 simple groups:

| Group | Meaning |
|---|---|
| P/LP | Pathogenic or likely pathogenic (causes disease) |
| VUS | Uncertain significance (not enough evidence) |
| B/LB | Benign or likely benign (harmless) |
| Conflicting | Labs disagree |
| Other | Drug response, risk factor, etc. |

In [ ]:
def sig_group(label):
    if pd.isna(label):
        return "Other"
    s = str(label).lower()
    if "conflicting" in s:
        return "Conflicting"
    has_path = "pathogenic" in s
    has_ben = "benign" in s
    if has_path and not has_ben:
        return "P/LP"
    if has_ben and not has_path:
        return "B/LB"
    if "uncertain" in s:
        return "VUS"
    return "Other"

# Map unique labels once (fast), then apply to all rows
label_map = {lab: sig_group(lab) for lab in df["ClinicalSignificance"].dropna().unique()}
df["SigGroup"] = df["ClinicalSignificance"].map(label_map).fillna("Other")

print("Unique raw labels:", df["ClinicalSignificance"].nunique())
df["SigGroup"].value_counts()

### 3e · Turn review status into stars (0–4)
Stars show how much to trust a label.

In [ ]:
def stars(status):
    if pd.isna(status):
        return 0
    s = str(status).lower()
    if "practice guideline" in s:
        return 4
    if "expert panel" in s:
        return 3
    if "multiple submitters" in s and "no conflict" in s:
        return 2
    if "criteria provided" in s:  # single submitter, or conflicting
        return 1
    return 0

star_map = {st: stars(st) for st in df["ReviewStatus"].dropna().unique()}
df["Stars"] = df["ReviewStatus"].map(star_map).fillna(0).astype(int)
df["Stars"].value_counts().sort_index()

### 3f · Save memory: repeated text → category

In [ ]:
for c in ["Type", "ClinicalSignificance", "ReviewStatus", "Assembly", "Chromosome",
          "OriginSimple", "SigGroup"]:
    df[c] = df[c].astype("category")
df = df.drop(columns=["Assembly"])
print(f"Memory used: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

### 3g · Save the clean file (reload it in seconds next time)

In [ ]:
df.to_parquet(CLEAN_FILE, index=False)
!ls -lh "{CLEAN_FILE}"
# Next session you can skip Steps 1–3 and run:
# df = pd.read_parquet(CLEAN_FILE)

## Step 4 · Analyze and visualize

Colors match the deck: red = P/LP, amber = VUS, green = B/LB, slate = Conflicting.

In [ ]:
COLORS = {"P/LP": "#E4572E", "VUS": "#F3A712", "B/LB": "#3BB273",
          "Conflicting": "#5B6C8F", "Other": "#B8C4D0"}
ORDER = ["P/LP", "VUS", "B/LB", "Conflicting", "Other"]

### 4a · How much is still unknown? (share of each group)

In [ ]:
share = df["SigGroup"].value_counts(normalize=True).reindex(ORDER).fillna(0) * 100
ax = share.plot.bar(color=[COLORS[g] for g in ORDER], figsize=(8, 4))
ax.set_ylabel("% of variants"); ax.set_xlabel("")
ax.set_title("Clinical significance of ClinVar variants (GRCh38)")
for i, v in enumerate(share):
    ax.text(i, v + 0.5, f"{v:.1f}%", ha="center")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

### 4b · Which genes carry the most disease-causing variants?
Some rows list several genes (`BRCA1;NBR2`), so we split them first.

In [ ]:
genes = (df.loc[df["SigGroup"] == "P/LP", ["GeneSymbol"]]
           .dropna()
           .assign(GeneSymbol=lambda d: d["GeneSymbol"].str.split(";"))
           .explode("GeneSymbol"))
top_genes = genes["GeneSymbol"].value_counts().head(15).sort_values()
ax = top_genes.plot.barh(color=COLORS["P/LP"], figsize=(8, 5))
ax.set_title("Top 15 genes by number of P/LP variants"); ax.set_xlabel("Variants")
plt.tight_layout(); plt.show()

### 4c · Where do labs disagree most?

In [ ]:
conf = (df.loc[df["SigGroup"] == "Conflicting", ["GeneSymbol"]].dropna()
          .assign(GeneSymbol=lambda d: d["GeneSymbol"].str.split(";"))
          .explode("GeneSymbol"))
top_conf = conf["GeneSymbol"].value_counts().head(15).sort_values()
ax = top_conf.plot.barh(color=COLORS["Conflicting"], figsize=(8, 5))
ax.set_title("Top 15 genes by conflicting classifications"); ax.set_xlabel("Variants")
plt.tight_layout(); plt.show()

### 4d · How trustworthy are the labels? (review stars)

In [ ]:
star_counts = df["Stars"].value_counts().sort_index()
ax = star_counts.plot.bar(color="#0FA3B1", figsize=(7, 4))
ax.set_title("Review status (0 = no evidence given, 4 = practice guideline)")
ax.set_xlabel("Stars"); ax.set_ylabel("Variants")
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

pd.crosstab(df["Stars"], df["SigGroup"], normalize="index").reindex(columns=ORDER).mul(100).round(1)

### 4e · Does the type of DNA change matter?

In [ ]:
top_types = df["Type"].value_counts().head(6).index
sub = df[df["Type"].isin(top_types)]
ct = (pd.crosstab(sub["Type"].astype(str), sub["SigGroup"], normalize="index")
        .reindex(columns=ORDER).fillna(0) * 100)
ax = ct.plot.barh(stacked=True, color=[COLORS[g] for g in ORDER], figsize=(9, 4.5))
ax.set_title("Significance by variant type (% of each type)"); ax.set_xlabel("%")
ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left"); plt.tight_layout(); plt.show()

### 4f · How knowledge changes over time

In [ ]:
yearly = (df.dropna(subset=["EvalYear"])
            .query("EvalYear >= 2010")
            .groupby(["EvalYear", "SigGroup"], observed=True).size()
            .unstack(fill_value=0).reindex(columns=ORDER, fill_value=0))
ax = yearly.plot(color=[COLORS[g] for g in ORDER], figsize=(9, 4.5), marker="o")
ax.set_title("Variants by year of last evaluation"); ax.set_xlabel("Year"); ax.set_ylabel("Variants")
plt.tight_layout(); plt.show()

### 4g · Most common linked conditions (P/LP only)

In [ ]:
phen = (df.loc[df["SigGroup"] == "P/LP", ["PhenotypeList"]].dropna()
          .assign(PhenotypeList=lambda d: d["PhenotypeList"].str.split("|"))
          .explode("PhenotypeList"))
phen = phen[~phen["PhenotypeList"].str.lower().isin(["not provided", "not specified"])]
top_phen = phen["PhenotypeList"].value_counts().head(15).sort_values()
ax = top_phen.plot.barh(color=COLORS["P/LP"], figsize=(9, 5.5))
ax.set_title("Top 15 conditions linked to P/LP variants"); ax.set_xlabel("Variants")
plt.tight_layout(); plt.show()

## Step 5 · Export summary tables (for the slides)

In [ ]:
summary = {
    "significance_share": share.round(2).to_frame("percent"),
    "top_plp_genes": top_genes.sort_values(ascending=False).to_frame("variants"),
    "top_conflicting_genes": top_conf.sort_values(ascending=False).to_frame("variants"),
    "review_stars": star_counts.to_frame("variants"),
}
for name, table in summary.items():
    table.to_csv(f"{DATA_DIR}/{name}.csv")
print("Saved:", os.listdir(DATA_DIR))